# 06 — Baseline model selection

## Goal

Select the baseline architecture using matched Davis and KIBA validation evidence across random, cold-drug, and cold-target settings. Test performance is not a selection criterion.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import EXPERIMENT_LOG_PATH

RUN_SELECTION = False
print("Experiment log:", EXPERIMENT_LOG_PATH)


## Steps


In [ ]:
if RUN_SELECTION:
    import pandas as pd

    records = pd.read_csv(EXPERIMENT_LOG_PATH)
    rank_models = require_api(
        "src.training", "rank_baseline_models_by_validation"
    )
    select_model = require_api(
        "src.training", "select_best_baseline_architecture"
    )
    selection = rank_models(
        records, require_complete_evidence=True
    )
    selected_model = select_model(
        records, require_complete_evidence=True
    )
    print(selection.to_string(index=False))
    print("Selected baseline:", selected_model)
else:
    print("Selection skipped: RUN_SELECTION is False.")


## Checks

One complete seed is sufficient for development; additional complete seeds strengthen the final evidence. For every included seed, the strict source helper requires GCN/GAT/GIN across all six Davis/KIBA split cells with matched protocols, and unresolved rank ties fail closed. Do not average Davis pKd errors directly with KIBA-score errors as though they share a scale; validation ranks provide a scale-aware cross-dataset summary.

## Next steps

Freeze the selection rationale and checkpoint references before advanced-model comparison or AML screening.
